In [1]:
from bayesianquilts.jax.parameter import Interactions, Decomposed
import jax
import jax.numpy as jnp

Let's consider extending the linear regression problem

$$ y = \vec{x} \cdot \vec{\beta} $$

so that $\beta$ can vary. Let's assume that $\vec{\beta}$ is $100$ dimensional

Let's say you have some number of **discrete** factors over which you would like to vary $\beta$, for example, you might have sex, race, and whether one is currently a smoker. Let's assume you have 5 possible values for race, two for sex, and two for smoker status. Then you have $2\times 5\times 2=20$ different possible groups over which to fit your model, and $20\times 100=2000$ total regression parameters.

A common approach is to divide the data into these 20 groups, and fit 20 separate models. One can see who this procedure would be statistically problematic  already with this simple example. We would like to define a way of regularizing such a problem.

# Parameter decomposition method

In dividing the data into 20 groups, one doesn't allow the groups to share information. Instead, let's consider decomposing the parameter $\beta$ in terms of the order of interactions, so $\beta=\beta_0 + \beta_{sex}+\beta_{race}+\beta_{smoking} + \beta_{sex, gender} + \ldots $

## WHY??

The reason we  want to do this is because we can increase the strength of the regularization for higher order terms, in effect more-strongly forcing more of the higher-order contributions to zero by default. What this procedure does is partially pool the information in the dataset so that the model is effectively lower-order except in places where the data supports high-order contributions.

The `Interaction` and `Decomposed` classes work together in creating this decomposition:

In [2]:
interaction = Interactions(
    [
        ('sex', 2),
        ('race', 5),
        ('smoking', 2)
    ], exclusions=[]
    )

print(interaction)

Interaction dimensions: [Dimension(name=sex, cardinality=2), Dimension(name=race, cardinality=5), Dimension(name=smoking, cardinality=2)]


In [3]:
beta = Decomposed(
    param_shape=[100],
    interactions=interaction,
    name='beta'
)
print(beta)

Parameter shape: [100] 
Interaction dimensions: [Dimension(name=sex, cardinality=2), Dimension(name=race, cardinality=5), Dimension(name=smoking, cardinality=2)] 
Component tensors: 8 
Effective parameter cardinality: 2000 
Actual parameter cardinality: 5400



We have created a representation for $\vec{\beta}$ that consists of eight component parameters:

In [4]:
for name, shape in beta._tensor_part_shapes.items():
    print(f"  {name}: {list(shape)}")

  beta__: [1, 1, 1, 100]
  beta__smoking: [1, 1, 2, 100]
  beta__race: [1, 5, 1, 100]
  beta__race_smoking: [1, 5, 2, 100]
  beta__sex: [2, 1, 1, 100]
  beta__sex_smoking: [2, 1, 2, 100]
  beta__sex_race: [2, 5, 1, 100]
  beta__sex_race_smoking: [2, 5, 2, 100]


The `Decomposed` class created the constituent parameters and initialized them to a value which is default of $\vec{0}$.

We can also choose to exclude certain interactions. Let's say we think the model should exclude the interaction between race and sex not qualified by smoking:

In [5]:
interaction = Interactions(
    [
        ('sex', 2),
        ('race', 5),
        ('smoking', 2)
    ], exclusions=[('race', 'sex')]
    )

beta = Decomposed(
    param_shape=[100],
    interactions=interaction,
    name='beta'
)
print(beta)
for name, shape in beta._tensor_part_shapes.items():
    print(f"  {name}: {list(shape)}")

Parameter shape: [100] 
Interaction dimensions: [Dimension(name=sex, cardinality=2), Dimension(name=race, cardinality=5), Dimension(name=smoking, cardinality=2)] 
Component tensors: 7 
Effective parameter cardinality: 2000 
Actual parameter cardinality: 4400

  beta__: [1, 1, 1, 100]
  beta__smoking: [1, 1, 2, 100]
  beta__race: [1, 5, 1, 100]
  beta__race_smoking: [1, 5, 2, 100]
  beta__sex: [2, 1, 1, 100]
  beta__sex_smoking: [2, 1, 2, 100]
  beta__sex_race_smoking: [2, 5, 2, 100]


You see that the number of constituent tensors is now $7$. In practice, we might wish to exclude higher order terms in order to save memory.

Now we have created the decomposition. Let's use the decomposition. Suppose I have a sample of people:

1. sex=1, race=1, smoking=1
2. sex=0, race=3, smoking=0
3. sex=0, race=2, smoking=1
4. sex=1, race=0, smoking=1

 and I want to retrieve the effect values of $\vec{\beta}$ for these people. The `Decomposed` class takes care of this lookup

In [6]:
indices = [
    [1, 1, 1],
    [0, 3, 0],
    [0, 2, 1],
    [1, 0, 1]
]

beta_effective = beta.lookup(indices)
print(beta_effective.shape)



(4, 1, 100)


You see that I have $4 \times 100$ values that are returned, where each row corresponds to the regression parameter vector for each of the people.

# Parameter batches

TFP works on parameters in sample batches. The `Decomposed` class handles batching. Let's generate a batch of size $5$ of the component tensors that add to $\vec{\beta}$:

In [7]:
tensors, tensor_names, tensor_shapes = beta.generate_tensors(batch_shape=[5])
print([f"{k}: {v.shape}" for k, v in tensors.items()])

['beta__: (5, 1, 100)', 'beta__smoking: (5, 2, 100)', 'beta__race: (5, 5, 100)', 'beta__race_smoking: (5, 10, 100)', 'beta__sex: (5, 2, 100)', 'beta__sex_smoking: (5, 4, 100)', 'beta__sex_race_smoking: (5, 20, 100)']


now let's query the batched parameter to get batched effective values:

In [8]:
beta.set_params(tensors)
effective_batched = beta.lookup(indices)
print(effective_batched.shape)


(5, 4, 1, 100)


So you see that for each of the 4 people we have a batch of size 5, of the vector $\vec{\beta}$.

# Interaction order structure

Each component in the decomposition has an *interaction order* — the number
of variables it depends on. The `Decomposed` class provides methods to
inspect this structure:


In [9]:
print(f"Maximum interaction order: {beta.max_order()}\n")

for order in range(beta.max_order() + 1):
    components = beta.components_at_order(order)
    print(f"Order {order}: {components}")
    for c in components:
        print(f"  {c} depends on: {beta._tensor_part_interactions[c]}")

Maximum interaction order: 3

Order 0: ['beta__']
  beta__ depends on: ()
Order 1: ['beta__smoking', 'beta__race', 'beta__sex']
  beta__smoking depends on: ('smoking',)
  beta__race depends on: ('race',)
  beta__sex depends on: ('sex',)
Order 2: ['beta__race_smoking', 'beta__sex_smoking']
  beta__race_smoking depends on: ('race', 'smoking')
  beta__sex_smoking depends on: ('sex', 'smoking')
Order 3: ['beta__sex_race_smoking']
  beta__sex_race_smoking depends on: ('sex', 'race', 'smoking')


# Sparsity assessment

In practice, higher-order interaction terms are often driven to zero by
regularization (e.g., horseshoe priors). The `Decomposed` class can assess
which components are effectively zero, using several methods:

- **`relative_norm`**: component norm relative to the global term
- **`absolute_norm`**: absolute L2 norm
- **`snr`**: signal-to-noise ratio (for surrogate params with loc/scale)

Let's simulate a scenario where the global and sex effects are strong,
but smoking and race effects are weak:

In [10]:
# Create a fresh decomposition (no exclusions this time)
interaction_full = Interactions(
    [('sex', 2), ('race', 5), ('smoking', 2)]
)
beta = Decomposed(param_shape=[100], interactions=interaction_full, name='beta')

# Simulate: global and sex effects are strong, others are weak
key = jax.random.PRNGKey(42)
keys = jax.random.split(key, 8)

beta._tensor_parts['beta__'] = jax.random.normal(keys[0], beta._tensor_parts['beta__'].shape)
beta._tensor_parts['beta__sex'] = 0.5 * jax.random.normal(keys[1], beta._tensor_parts['beta__sex'].shape)
beta._tensor_parts['beta__race'] = 0.01 * jax.random.normal(keys[2], beta._tensor_parts['beta__race'].shape)
# Leave smoking, and all higher-order terms, at zero

# Check norms
norms = beta.component_norms(beta._tensor_parts)
print("Component L2 norms:")
for name in sorted(norms):
    order = beta.component_order(name)
    print(f"  {name} (order {order}): {norms[name]:.4f}")

Component L2 norms:
  beta__ (order 0): 9.5197
  beta__race (order 1): 0.2192
  beta__race_smoking (order 2): 0.0000
  beta__sex (order 1): 6.8272
  beta__sex_race (order 2): 0.0000
  beta__sex_race_smoking (order 3): 0.0000
  beta__sex_smoking (order 2): 0.0000
  beta__smoking (order 1): 0.0000


In [11]:
# Identify sparse components (relative norm < 10% of global)
sparse = beta.sparse_components(beta._tensor_parts, threshold=0.1, method="relative_norm")
print(f"Sparse components (relative norm < 0.1): {sorted(sparse)}")
print(f"Non-sparse: {sorted(set(beta._tensor_parts.keys()) - sparse)}")

Sparse components (relative norm < 0.1): ['beta__race', 'beta__race_smoking', 'beta__sex_race', 'beta__sex_race_smoking', 'beta__sex_smoking', 'beta__smoking']
Non-sparse: ['beta__', 'beta__sex']


# Hereditary exclusions

The **hereditary principle** says: if a lower-order component is sparse (driven
to zero), then all higher-order interactions *containing* those same variables
cannot possibly be important either.

For example, if `beta__smoking` is sparse (smoking has no main effect), then
`beta__sex_smoking`, `beta__race_smoking`, and `beta__sex_race_smoking` should
all be excluded from training — they would waste parameters on interactions
that the data doesn't support.

This is the key insight behind **sparse staged training**: train components
progressively from low to high order, prune sparse branches, and skip
their descendants.

In [12]:
excluded = beta.hereditary_exclusions(sparse)
print(f"Sparse components: {sorted(sparse)}")
print(f"Excluded descendants: {sorted(excluded)}")
print()

# What remains active?
all_components = set(beta._tensor_parts.keys())
active = all_components - sparse - excluded
pruned = (sparse | excluded) - {'beta__'}  # global is never pruned
print(f"Active components after pruning: {sorted(active)}")
print(f"  ({len(active)} of {len(all_components)} components)")
print(f"  Pruned: {len(pruned)} components")

Sparse components: ['beta__race', 'beta__race_smoking', 'beta__sex_race', 'beta__sex_race_smoking', 'beta__sex_smoking', 'beta__smoking']
Excluded descendants: ['beta__race_smoking', 'beta__sex_race', 'beta__sex_race_smoking', 'beta__sex_smoking']

Active components after pruning: ['beta__', 'beta__sex']
  (2 of 8 components)
  Pruned: 6 components


# Sparse staged training

The quilted models (`LogisticBayesianquilt`, `RegressionBayesianquilt`,
`LinearBayesianquilt`) use `staged_fit()` to train progressively by
interaction order. Here's how it works:

**For each order 0, 1, 2, ..., max_order:**
1. Activate new components at this order
2. Train them via ADVI (previously-trained components are frozen as point estimates)
3. Assess sparsity — identify components driven to zero
4. Apply hereditary exclusions — skip descendants of sparse components

**Final stage:** Unfreeze all active components for joint optimization at
full precision.

Let's walk through what a staged training run would look like with our
3-factor example:

In [13]:
# Simulate the staged training decision process
# (using the same beta with simulated post-training norms from above)

interaction_full = Interactions(
    [('sex', 2), ('race', 5), ('smoking', 2)]
)
beta = Decomposed(param_shape=[100], interactions=interaction_full, name='beta')

# After "training" stage 0 — only the global component was active
key = jax.random.PRNGKey(42)
keys = jax.random.split(key, 8)
beta._tensor_parts['beta__'] = jax.random.normal(keys[0], beta._tensor_parts['beta__'].shape)
print("=== Stage 0 (order 0): Train global component ===")
print(f"  Active: {beta.components_at_order(0)}")
print(f"  Global norm: {float(jnp.linalg.norm(beta._tensor_parts['beta__'])):.4f}")
print()

# After "training" stage 1 — sex, race, smoking main effects
# Simulate: sex effect is strong, race/smoking are weak
beta._tensor_parts['beta__sex'] = 0.5 * jax.random.normal(keys[1], beta._tensor_parts['beta__sex'].shape)
beta._tensor_parts['beta__race'] = 0.01 * jax.random.normal(keys[2], beta._tensor_parts['beta__race'].shape)
beta._tensor_parts['beta__smoking'] = 0.005 * jax.random.normal(keys[3], beta._tensor_parts['beta__smoking'].shape)

print("=== Stage 1 (order 1): Train main effects ===")
print(f"  Active: {beta.components_at_order(1)}")
norms = beta.component_norms(beta._tensor_parts)
global_norm = norms['beta__']
for name in beta.components_at_order(1):
    rel = norms[name] / (global_norm + 1e-10)
    status = "SPARSE" if rel < 0.1 else "active"
    print(f"  {name}: norm={norms[name]:.4f}, relative={rel:.4f} [{status}]")

sparse_1 = beta.sparse_components(beta._tensor_parts, threshold=0.1, method="relative_norm")
sparse_1 = {s for s in sparse_1 if beta.component_order(s) == 1}
excluded = beta.hereditary_exclusions(sparse_1)
print(f"\n  Sparse at order 1: {sorted(sparse_1)}")
print(f"  Excluding descendants: {sorted(excluded)}")
print()

# Stage 2 — 2-way interactions, but only those not excluded
print("=== Stage 2 (order 2): Train 2-way interactions ===")
order2 = beta.components_at_order(2)
active_order2 = [c for c in order2 if c not in excluded]
skipped_order2 = [c for c in order2 if c in excluded]
print(f"  Would train: {active_order2}")
print(f"  Skipped (excluded): {skipped_order2}")
print()

# Stage 3 — 3-way interaction
print("=== Stage 3 (order 3): Train 3-way interaction ===")
order3 = beta.components_at_order(3)
active_order3 = [c for c in order3 if c not in excluded]
skipped_order3 = [c for c in order3 if c in excluded]
print(f"  Would train: {active_order3}")
print(f"  Skipped (excluded): {skipped_order3}")
print()

total = len(beta._tensor_parts)
n_active = len(active_order2) + len(active_order3) + len(beta.components_at_order(0)) + \
           len([c for c in beta.components_at_order(1) if c not in sparse_1])
n_pruned = total - n_active
print(f"=== Summary ===")
print(f"  Total components: {total}")
print(f"  Active after pruning: {n_active}")
print(f"  Pruned (sparse + excluded): {n_pruned}")
print(f"  Savings: {n_pruned/total*100:.0f}% fewer components to train")

=== Stage 0 (order 0): Train global component ===
  Active: ['beta__']
  Global norm: 9.5197

=== Stage 1 (order 1): Train main effects ===
  Active: ['beta__smoking', 'beta__race', 'beta__sex']
  beta__smoking: norm=0.0664, relative=0.0070 [SPARSE]
  beta__race: norm=0.2192, relative=0.0230 [SPARSE]
  beta__sex: norm=6.8272, relative=0.7172 [active]

  Sparse at order 1: ['beta__race', 'beta__smoking']
  Excluding descendants: ['beta__race_smoking', 'beta__sex_race', 'beta__sex_race_smoking', 'beta__sex_smoking']

=== Stage 2 (order 2): Train 2-way interactions ===
  Would train: []
  Skipped (excluded): ['beta__race_smoking', 'beta__sex_smoking', 'beta__sex_race']

=== Stage 3 (order 3): Train 3-way interaction ===
  Would train: []
  Skipped (excluded): ['beta__sex_race_smoking']

=== Summary ===
  Total components: 8
  Active after pruning: 2
  Pruned (sparse + excluded): 6
  Savings: 75% fewer components to train


# Surrogate key mapping

During ADVI training, each decomposition component `beta__sex` gets surrogate
parameters like `beta__sex\loc` and `beta__sex\scale`. The static helper
methods on `Decomposed` map between component names and surrogate keys,
which is how `staged_fit()` constructs `optimize_keys` for each stage:

In [14]:
# Simulate what surrogate parameter keys look like
sep = "\\"  # backslash separator used in surrogate param names
all_surrogate_keys = []
for comp_name in sorted(beta._tensor_parts.keys()):
    all_surrogate_keys.append(f"{comp_name}{sep}loc")
    all_surrogate_keys.append(f"{comp_name}{sep}scale")
# Add auxiliary (non-component) keys
all_surrogate_keys.extend([f"tau{sep}loc", f"tau{sep}scale",
                           f"lambda_j{sep}loc", f"lambda_j{sep}scale",
                           f"c2{sep}loc", f"c2{sep}scale"])

all_component_names = set(beta._tensor_parts.keys())

# Which keys belong to active components only?
active_components = {'beta__', 'beta__sex'}  # just global + sex
active_keys = Decomposed.surrogate_keys_for_components(active_components, all_surrogate_keys)
print("Active component keys (for optimize_keys):")
for k in active_keys:
    print(f"  {k}")

# Auxiliary keys (horseshoe params) — always optimized
aux_keys = Decomposed.non_component_keys(all_component_names, all_surrogate_keys)
print(f"\nAuxiliary keys (always optimized):")
for k in aux_keys:
    print(f"  {k}")

Active component keys (for optimize_keys):
  beta__\loc
  beta__\scale
  beta__sex\loc
  beta__sex\scale

Auxiliary keys (always optimized):
  tau\loc
  tau\scale
  lambda_j\loc
  lambda_j\scale
  c2\loc
  c2\scale


# Using staged_fit with quilted models

The quilted models (`LogisticBayesianquilt`, `RegressionBayesianquilt`,
`LinearBayesianquilt`) now inherit from `QuiltedBayesianModel` and their
`fit()` method delegates to `staged_fit()`. You can call it directly for
more control:

```python
from bayesianquilts.predictors.regression.regression_bayesianquilt import RegressionBayesianquilt

# Option 1: Use fit() with staged training (default)
model.fit(
    batched_data_factory=data_factory,
    batch_size=256,
    dataset_size=10000,
    warmup_max_order=3,          # train up to 3-way interactions
    sparsity_threshold=0.1,      # prune components < 10% of global norm
    sparsity_method="relative_norm",
    epochs_per_stage=25,         # epochs per warmup stage
    num_epochs=100,              # epochs for final joint training
)

# Option 2: Use staged_fit() directly for more control
model.staged_fit(
    batched_data_factory=data_factory,
    batch_size=256,
    dataset_size=10000,
    max_order=3,
    sparsity_threshold=0.1,
    sparsity_method="relative_norm",
    epochs_per_stage=25,
    final_epochs=100,
    freeze_between_stages=True,  # freeze as point estimates
    coarse_dtype=jnp.float32,    # lower precision for early stages
)

# After training, inspect what was pruned:
print(f"Active components: {model.active_components}")
print(f"Excluded components: {model.excluded_components}")
```

Setting `sparsity_threshold=0.0` disables pruning and reproduces the
original progressive warmup behavior (but with the fix for `optimize_keys`).